In [ ]:
# Cell 0: Install extra deps + GPU check
import subprocess, os, torch

# Do NOT reinstall PyTorch - Kaggle P100 already has correct cu128 version
subprocess.run([
    "pip", "install", "-q",
    "numpy", "scipy", "pandas", "scikit-learn", "tqdm", "pytorch_metric_learning"
], check=True)

assert torch.cuda.is_available(), "Enable GPU in notebook settings!"
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("Setup complete.")

In [ ]:
# Cell 1: Config + constants
import os, struct, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from collections import namedtuple
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    mpi_sampling_rate: int = 50
    mpi_window_sec: float = 3.0
    mpi_min_readings: int = 100
    mpi_n_channels: int = 18
    mpi_epochs: int = 30
    stationary_threshold: float = 0.01
    uv_sampling_rate: int = 50
    uv_window_sec: float = 1.0
    uv_augment_window_sec: float = 1.5
    uv_n_features: int = 22
    uv_n_channels_per_feature: int = 4
    uv_n_trials: int = 300
    uv_n_clusters: int = 6
    uv_total_users: int = 101
    uv_test_users: int = 11
    uv_baseline_n: int = 75
    uv_n_splits: List[int] = field(default_factory=lambda: [60, 65, 70, 75, 80, 85])
    baseline_lr: float = 1e-3
    finetune_lr: float = 1e-4
    baseline_epochs: int = 50
    finetune_epochs: int = 10
    batch_size: int = 64
    alpha_tm: float = 1.0
    supcon_temperature: float = 0.07
    n_training_runs: int = 5
    tar_threshold: float = 0.90
    target_far: float = 1 / 50000
    bootstrap_repeats: int = 5000
    far_sweep_steps: int = 100_000

cfg = Config()

FLAG_USER_PRESENT = "android.intent.action.USER_PRESENT"
FLAG_SCREEN_OFF   = "android.intent.action.SCREEN_OFF"
FLAG_SCREEN_ON    = "android.intent.action.SCREEN_ON"
RECORD_SIZE = 20

SENSOR_FILES = {
    "acc": "accel.txt", "grav": "gravity.txt", "gyro": "gyro.txt",
    "lin": "linAccel.txt", "mag": "MagneticField.txt", "rot": "Rotation.txt",
}
SCREEN_FILE = "screen.txt"
SENSOR_COLS = {n: [f"{n}_X", f"{n}_Y", f"{n}_Z"] for n in SENSOR_FILES}
ALL_SENSOR_COLS = (SENSOR_COLS["acc"] + SENSOR_COLS["grav"] + SENSOR_COLS["gyro"] +
                   SENSOR_COLS["lin"] + SENSOR_COLS["mag"] + SENSOR_COLS["rot"])

BASE = "/kaggle/input/datasets/djaarf"
MPI_DIRS = [
    f"{BASE}/motionid-imu-all-motions-part1/IMU_all_motions_part1",
    f"{BASE}/motionid-imu-all-motions-part2/IMU_all_motions_part2",
    f"{BASE}/motionid-imu-all-motions-part3/IMU_all_motions_part3",
]
UV_DIR = f"{BASE}/motionid-imu-specific-motion/IMU_specific_motion/train_val_test"
PROCESSED_MPI = "/kaggle/working/data/mpi/processed"
PROCESSED_UV  = "/kaggle/working/data/uv/processed"
CKPT_DIR = "/kaggle/working/models/checkpoints"
RESULTS_DIR = "/kaggle/working/evaluation"
os.makedirs(PROCESSED_MPI, exist_ok=True)
os.makedirs(PROCESSED_UV, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Config loaded. Device: {device}")

In [ ]:
# Cell 2: Numpy-only sensor reader (low memory, fast)
# Loads each sensor file as numpy array, not pandas DataFrame.
# Aligns sensors by timestamp using np.interp.

def read_sensor_numpy(path):
    with open(path, "rb") as f:
        data = f.read()
    n = len(data) // 20
    if n == 0: return np.empty((0, 4), dtype=np.float64)
    arr = np.frombuffer(data, dtype=np.uint8).reshape(n, 20)
    ts = np.zeros(n, dtype=np.float64)
    for i in range(8):
        ts += arr[:, i].astype(np.float64) * (2 ** (56 - 8 * i))
    xyz = arr[:, 8:20].view(np.float32).reshape(n, 3)
    out = np.column_stack([ts, xyz])
    return out[out[:, 0].argsort()]

def read_screen(path):
    rows = []
    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                try: rows.append((int(parts[0]), parts[1]))
                except ValueError: continue
    return rows

def load_all_sensors_numpy(session_dir):
    sensors = {}
    for name, fname in SENSOR_FILES.items():
        fpath = os.path.join(session_dir, fname)
        if not os.path.exists(fpath): return None
        arr = read_sensor_numpy(fpath)
        if len(arr) == 0: return None
        sensors[name] = arr
    return sensors

def get_aligned_window(sensors, t_start, t_end):
    # Use acc timestamps as reference
    acc = sensors["acc"]
    idx_start = np.searchsorted(acc[:, 0], t_start, side="left")
    idx_end   = np.searchsorted(acc[:, 0], t_end,   side="right")
    if idx_end - idx_start < 10: return None
    acc_ts = acc[idx_start:idx_end, 0]
    result_cols = []
    for name in ["acc", "grav", "gyro", "lin", "mag", "rot"]:
        s = sensors[name]
        s_idx = np.searchsorted(s[:, 0], t_start, side="left")
        s_ie  = np.searchsorted(s[:, 0], t_end,   side="right")
        if s_ie - s_idx < 5: return None
        s_ts = s[s_idx:s_ie, 0]
        for col in range(1, 4):
            result_cols.append(np.interp(acc_ts, s_ts, s[s_idx:s_ie, col]))
    return np.column_stack(result_cols)

print("Numpy reader ready (no pandas for sensor data).")


In [ ]:
# Cell 3: MPI Preprocessing (numpy arrays, no pandas merge)
from scipy import interpolate
from tqdm import tqdm

def discover_sessions(input_dirs):
    sessions = []
    for root in input_dirs:
        if not os.path.exists(root): continue
        for user in sorted(os.listdir(root)):
            up = os.path.join(root, user)
            if not os.path.isdir(up): continue
            for phone in sorted(os.listdir(up)):
                pp = os.path.join(up, phone)
                if not os.path.isdir(pp): continue
                for session in sorted(os.listdir(pp)):
                    sp = os.path.join(pp, session)
                    if os.path.isdir(sp): sessions.append((user, phone, sp))
    return sessions

def normalize_length(sample, target_len=150):
    n, c = sample.shape
    if n == target_len: return sample.T.astype(np.float32)
    if n < target_len:
        x_old = np.linspace(0, 1, n)
        x_new = np.linspace(0, 1, target_len)
        out = np.zeros((target_len, c), dtype=np.float32)
        for i in range(c):
            out[:, i] = np.interp(x_new, x_old, sample[:, i])
        return out.T
    return sample[-target_len:].T.astype(np.float32)

def process_session(uid, did, session_dir):
    target_len = int(cfg.mpi_sampling_rate * cfg.mpi_window_sec)
    window_ms  = int(cfg.mpi_window_sec * 1000)

    screen_path = os.path.join(session_dir, SCREEN_FILE)
    if not os.path.exists(screen_path): return None, None
    screen_events = read_screen(screen_path)
    if not screen_events: return None, None

    sensors = load_all_sensors_numpy(session_dir)
    if sensors is None: return None, None

    pos = []
    for ts, event in screen_events:
        if event != FLAG_USER_PRESENT: continue
        w = get_aligned_window(sensors, ts - window_ms, ts)
        if w is not None and len(w) >= cfg.mpi_min_readings:
            pos.append(w.astype(np.float32))

    neg, max_read_ms = [], 60000
    screen_ts = [t for t, _ in screen_events]
    screen_ev = [e for _, e in screen_events]
    for i, (off_ts, event) in enumerate(screen_events):
        if event != FLAG_SCREEN_OFF: continue
        end_ts = None
        for j in range(i+1, len(screen_events)):
            if screen_ev[j] in (FLAG_SCREEN_ON, FLAG_USER_PRESENT):
                end_ts = screen_ts[j]; break
        if end_ts is None: continue
        effective_end = end_ts - window_ms
        if effective_end <= off_ts: continue
        read_start = max(off_ts, effective_end - max_read_ms)
        if effective_end - read_start < window_ms: continue
        w = get_aligned_window(sensors, read_start, effective_end)
        if w is None or len(w) < cfg.mpi_min_readings: continue
        lin = w[:, 9:12]  # lin_X, lin_Y, lin_Z
        if np.all(np.abs(lin) < cfg.stationary_threshold): continue
        neg.append(w.astype(np.float32))

    if len(pos) < 10 or len(neg) < 10: return None, None
    X = np.stack([normalize_length(s, target_len) for s in pos + neg])
    y = np.array([1]*len(pos) + [0]*len(neg), dtype=np.int64)
    return X, y

sessions = discover_sessions(MPI_DIRS)
print(f"Found {len(sessions)} MPI sessions")
rows = []
for uid, did, sdir in tqdm(sessions, desc="MPI"):
    X, y = process_session(uid, did, sdir)
    key = f"{uid}_{did}"
    if X is None:
        rows.append({"user_id": uid, "device_id": did, "n_pos": 0, "n_neg": 0, "status": "N/A"})
    else:
        np.savez(os.path.join(PROCESSED_MPI, f"{key}.npz"), X=X, y=y)
        n_pos, n_neg = int((y==1).sum()), int((y==0).sum())
        tqdm.write(f"  {uid}/{did}: X={X.shape} pos={n_pos} neg={n_neg}")
        rows.append({"user_id": uid, "device_id": did, "n_pos": n_pos, "n_neg": n_neg, "status": "OK"})
mf = pd.DataFrame(rows)
mf.to_csv(os.path.join(PROCESSED_MPI, "manifest.csv"), index=False)
print(f"MPI done. Valid: {(mf.status=='OK').sum()}/{len(mf)}")


In [ ]:
# Cell 4: UV Preprocessing (binary search per event)
from scipy.spatial.transform import Rotation as R
from tqdm import tqdm

def rotate_to_earth(v, rot_vec):
    return R.from_rotvec(rot_vec).apply(v).astype(np.float32)

def extract_window(merged_df, unlock_ts):
    n = int(cfg.uv_window_sec * cfg.uv_sampling_rate)
    window_ms = int(cfg.uv_window_sec * 1000)
    sub = merged_df[(merged_df["timestamp"] <= unlock_ts) &
                    (merged_df["timestamp"] > unlock_ts - window_ms)
                    ].sort_values("timestamp").tail(n)
    return sub.reset_index(drop=True) if len(sub) == n else None

def pad4(x, T):
    out = np.zeros((4, T), dtype=np.float32); out[:3, :] = x.T; return out

def diff(x): return np.concatenate([x[:1], np.diff(x, axis=0)], axis=0)

def compute_features(window):
    T = len(window)
    acc = window[["acc_X","acc_Y","acc_Z"]].values.astype(np.float32)
    grav = window[["grav_X","grav_Y","grav_Z"]].values.astype(np.float32)
    gyro = window[["gyro_X","gyro_Y","gyro_Z"]].values.astype(np.float32)
    mag = window[["mag_X","mag_Y","mag_Z"]].values.astype(np.float32)
    rot = window[["rot_X","rot_Y","rot_Z"]].values.astype(np.float32)
    lin_acc = acc - grav
    acc_rot = rotate_to_earth(acc, rot)
    gyro_rot = rotate_to_earth(gyro, rot)
    mag_rot = rotate_to_earth(mag, rot)
    return np.stack([
        pad4(acc, T), pad4(gyro, T), pad4(mag, T), pad4(lin_acc, T),
        pad4(acc_rot, T), pad4(gyro_rot, T), pad4(mag_rot, T),
        pad4(diff(acc), T), pad4(diff(gyro), T), pad4(diff(mag), T),
        pad4(diff(acc_rot), T), pad4(diff(gyro_rot), T), pad4(diff(mag_rot), T),
        pad4(np.cumsum(acc, axis=0), T), pad4(np.cumsum(gyro, axis=0), T),
        pad4(np.cumsum(mag, axis=0), T), pad4(np.cumsum(acc_rot, axis=0), T),
        pad4(np.cumsum(gyro_rot, axis=0), T), pad4(np.cumsum(mag_rot, axis=0), T),
        pad4(diff(lin_acc), T), pad4(np.cumsum(lin_acc, axis=0), T),
        pad4(rot, T),
    ], axis=0)

assert os.path.exists(UV_DIR), f"UV not found: {UV_DIR}"
user_dirs = sorted(d for d in os.listdir(UV_DIR) if os.path.isdir(os.path.join(UV_DIR, d)))
print(f"Found {len(user_dirs)} UV users")

for uid in tqdm(user_dirs, desc="UV"):
    base = os.path.join(UV_DIR, uid, "s20")
    if not os.path.exists(base): continue
    sessions = sorted(os.listdir(base))
    if not sessions: continue
    sdir = os.path.join(base, sessions[0])
    # Read screen first, then load sensors in that time range
    screen_path = os.path.join(sdir, SCREEN_FILE)
    screen_tmp = read_screen(screen_path)
    if len(screen_tmp) == 0: continue
    t_min = int(screen_tmp["timestamp"].min()) - 2000
    t_max = int(screen_tmp["timestamp"].max()) + 2000
    merged = read_sensors_window(sdir, t_start=t_min, t_end=t_max)
    if merged is None: tqdm.write(f"  {uid}: load failed"); continue
    screen_df = screen_tmp

    unlocks = screen_df[screen_df["event"] == FLAG_USER_PRESENT].sort_values("timestamp")
    feats, clusters, n_skip = [], [], 0
    for trial_idx, (_, row) in enumerate(unlocks.iterrows()):
        w = extract_window(merged, int(row["timestamp"]))
        if w is None: n_skip += 1; continue
        feats.append(compute_features(w))
        clusters.append(trial_idx // 50)
    if not feats: continue
    F = np.stack(feats, 0).astype(np.float32)
    C = np.array(clusters, dtype=np.int64)
    np.savez(os.path.join(PROCESSED_UV, f"{uid}.npz"), features=F, cluster_ids=C, user_id=uid)
    tqdm.write(f"  {uid}: {F.shape}")
done = len([f for f in os.listdir(PROCESSED_UV) if f.endswith(".npz")])
print(f"UV done. {done} users saved.")

In [ ]:
# Cell 5: Models + Losses

class UVBranch(nn.Module):
    def __init__(self, in_channels=4):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.conv3 = nn.Conv1d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool = nn.AdaptiveAvgPool1d(8)
        self.fc = nn.Linear(128 * 8, 256)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        return self.fc(self.pool(x).flatten(1))

class UVModel(nn.Module):
    def __init__(self, n_classes, n_features=22):
        super().__init__()
        self.n_features = n_features
        self.embed_dim = n_features * 256
        self.branches = nn.ModuleList([UVBranch(4) for _ in range(n_features)])
        self.head_a = nn.Linear(self.embed_dim, n_classes)
        self.siamese_proj = nn.Linear(self.embed_dim, 256)
        self.head_b = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 64))
    def _augment(self, x):
        T_t = int(cfg.uv_window_sec * cfg.uv_sampling_rate)
        T = x.size(-1)
        if T > T_t:
            start = torch.randint(0, T - T_t + 1, (1,)).item()
            x = x[..., start:start+T_t]
        return x + torch.randn_like(x) * 0.01
    def extract_embedding(self, x):
        return torch.cat([self.branches[i](x[:, i, :, :]) for i in range(self.n_features)], dim=1)
    def forward(self, x, augment=False):
        if augment: x = self._augment(x)
        emb = self.extract_embedding(x)
        logits = self.head_a(emb)
        siamese = F.normalize(self.siamese_proj(emb), dim=1)
        return logits, self.head_b(siamese)
    def get_siamese_embed(self, x):
        with torch.no_grad():
            return F.normalize(self.siamese_proj(self.extract_embedding(x)), dim=1)

class MPIModel(nn.Module):
    def __init__(self, n_channels=18, n_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_channels, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 128, 5, padding=2), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(8))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(256*8, 256), nn.ReLU(), nn.Linear(256, n_classes))
    def forward(self, x): return self.classifier(self.net(x))

class TripletMarginLoss(nn.Module):
    def __init__(self, margin=1.0): super().__init__(); self.margin = margin
    def forward(self, embeddings, labels):
        B = embeddings.size(0)
        dist = ((embeddings.unsqueeze(0) - embeddings.unsqueeze(1))**2).sum(2)
        leq = labels.unsqueeze(0) == labels.unsqueeze(1)
        eye = torch.eye(B, dtype=torch.bool, device=embeddings.device)
        losses = []
        for i in range(B):
            pm = leq[i] & ~eye[i]; nm = ~leq[i]
            if not pm.any() or not nm.any(): continue
            d_ap = dist[i][pm].max(); d_neg = dist[i][nm]
            semi = d_neg[(d_neg > d_ap) & (d_neg < d_ap + self.margin)]
            d_an = semi.min() if semi.numel() > 0 else d_neg.min()
            losses.append(F.relu(d_ap - d_an + self.margin))
        return torch.stack(losses).mean() if losses else torch.tensor(0.0, requires_grad=True, device=embeddings.device)

class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07): super().__init__(); self.temperature = temperature
    def forward(self, proj_embeddings, labels):
        N = proj_embeddings.size(0); B = labels.size(0)
        labels_rep = labels.repeat(2) if N == 2 * B else labels
        z = F.normalize(proj_embeddings, dim=1)
        sim = torch.mm(z, z.T) / self.temperature
        eye = torch.eye(N, dtype=torch.bool, device=z.device)
        pos_mask = (labels_rep.unsqueeze(0) == labels_rep.unsqueeze(1)) & ~eye
        sim = sim - sim.max(dim=1, keepdim=True).values.detach()
        denom = torch.exp(sim).masked_fill(eye, 0).sum(1, keepdims=True)
        log_prob = sim - torch.log(denom + 1e-8)
        n_pos = pos_mask.sum(1).float(); valid = n_pos > 0
        loss = -(log_prob * pos_mask.float()).sum(1)
        return (loss[valid] / n_pos[valid]).mean()

class TotalLoss(nn.Module):
    def __init__(self, alpha_tm=1.0, temperature=0.07):
        super().__init__()
        self.alpha_tm = alpha_tm
        self.ce = nn.CrossEntropyLoss()
        self.tm = TripletMarginLoss(1.0)
        self.sc = SupervisedContrastiveLoss(temperature)
    def forward(self, logits, proj_embeds, labels):
        lce = self.ce(logits, labels)
        ltm = self.tm(F.normalize(proj_embeds[:labels.size(0)], dim=1), labels)
        lsc = self.sc(proj_embeds, labels)
        return lce + self.alpha_tm * ltm + lsc, {"lce": lce.item(), "ltm": ltm.item(), "lsc": lsc.item()}

print("Models + losses ready.")

In [ ]:
# Cell 6: MPI Training
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, accuracy_score
from torch.utils.data import TensorDataset, DataLoader

def train_mpi_one(X, y, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    sss1 = StratifiedShuffleSplit(1, test_size=0.30, random_state=seed)
    idx_tr, idx_tmp = next(sss1.split(X, y))
    sss2 = StratifiedShuffleSplit(1, test_size=0.50, random_state=seed)
    idx_v, idx_te = next(sss2.split(X[idx_tmp], y[idx_tmp]))
    idx_v, idx_te = idx_tmp[idx_v], idx_tmp[idx_te]

    model = MPIModel(n_channels=X.shape[1], n_classes=2).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.baseline_lr)
    crit = nn.CrossEntropyLoss()

    def loader(Xa, ya, sh=True):
        return DataLoader(TensorDataset(torch.tensor(Xa), torch.tensor(ya, dtype=torch.long)),
                          batch_size=32, shuffle=sh)
    tr, va, te = loader(X[idx_tr], y[idx_tr]), loader(X[idx_v], y[idx_v], False), loader(X[idx_te], y[idx_te], False)

    best_auc, best_ckpt = -1, None
    for epoch in range(cfg.mpi_epochs):
        model.train()
        for Xb, yb in tr:
            opt.zero_grad(); crit(model(Xb.to(device)), yb.to(device)).backward(); opt.step()
        model.eval(); probs, ys = [], []
        with torch.no_grad():
            for Xb, yb in va:
                probs.extend(torch.softmax(model(Xb.to(device)), 1)[:, 1].cpu().numpy())
                ys.extend(yb.numpy())
        try: auc = roc_auc_score(ys, probs)
        except: auc = 0.5
        if auc > best_auc:
            best_auc = auc
            best_ckpt = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_ckpt); model.eval()
    preds, ys = [], []
    with torch.no_grad():
        for Xb, yb in te:
            preds.extend(model(Xb.to(device)).argmax(1).cpu().numpy())
            ys.extend(yb.numpy())
    return accuracy_score(ys, preds) * 100

mpi_files = [f for f in os.listdir(PROCESSED_MPI) if f.endswith(".npz")]
print(f"Training on {len(mpi_files)} MPI sessions, {cfg.n_training_runs} seeds each")

all_rows = []
for fname in mpi_files:
    data = np.load(os.path.join(PROCESSED_MPI, fname))
    X, y = data["X"], data["y"]
    uid, did = fname.replace(".npz", "").rsplit("_", 1)
    seed_accs = []
    for seed in range(cfg.n_training_runs):
        acc = train_mpi_one(X, y, seed)
        seed_accs.append(acc)
    all_rows.append({"user_id": uid, "device_id": did,
                     "mean_acc": round(np.mean(seed_accs), 2),
                     "std_acc": round(np.std(seed_accs), 2), "status": "OK"})
    print(f"  {uid}/{did}: {np.mean(seed_accs):.1f} +/- {np.std(seed_accs):.1f}%")

pd.DataFrame(all_rows).to_csv(os.path.join(RESULTS_DIR, "results_mpi.csv"), index=False)
print("MPI training done.")

In [ ]:
# Cell 7: UV Training (baseline + fine-tune + bootstrap)
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import accuracy_score
import copy

class UVDataset(Dataset):
    def __init__(self, feats_dict, label_map):
        self.samples, self.labels = [], []
        for uid, feats in feats_dict.items():
            lbl = label_map[uid]
            for t in feats:
                self.samples.append(t.astype(np.float32))
                self.labels.append(lbl)
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        return torch.tensor(self.samples[i]), torch.tensor(self.labels[i], dtype=torch.long)

def load_all_users():
    data = {}
    for f in sorted(os.listdir(PROCESSED_UV)):
        if not f.endswith(".npz"): continue
        try: uid = int(os.path.splitext(f)[0])
        except: continue
        data[uid] = np.load(os.path.join(PROCESSED_UV, f))["features"]
    return data

def split_attempts(feats, seed):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(feats.shape[0])
    n = len(idx); n_tr = int(n * 0.70); n_v = int(n * 0.15)
    return idx[:n_tr], idx[n_tr:n_tr+n_v], idx[n_tr+n_v:]

def score_verification(model, X_genuine, X_impostor):
    if len(X_genuine) == 0 or len(X_impostor) == 0: return np.array([]), np.array([])
    model.eval()
    with torch.no_grad():
        ge = model.get_siamese_embed(torch.tensor(X_genuine, dtype=torch.float32).to(device)).cpu().numpy()
        ie = model.get_siamese_embed(torch.tensor(X_impostor, dtype=torch.float32).to(device)).cpu().numpy()
    tmpl = ge.mean(0, keepdims=True); tmpl /= (np.linalg.norm(tmpl) + 1e-8)
    gn = ge / (np.linalg.norm(ge, axis=1, keepdims=True) + 1e-8)
    in_ = ie / (np.linalg.norm(ie, axis=1, keepdims=True) + 1e-8)
    return (gn @ tmpl.T).squeeze(), (in_ @ tmpl.T).squeeze()

def compute_far_at_tar(genuine, impostor, target_tar=0.90, n_steps=100_000):
    gen, imp = np.asarray(genuine, dtype=np.float64), np.asarray(impostor, dtype=np.float64)
    if len(gen) == 0 or len(imp) == 0: return 1.0, 0.0
    all_s = np.concatenate([gen, imp])
    for t in np.linspace(all_s.max(), all_s.min(), n_steps):
        if np.mean(gen >= t) >= target_tar: return float(np.mean(imp >= t)), float(t)
    return 1.0, float(all_s.min())

def bootstrap_far(genuine, impostor, n_repeats=5000, target_tar=0.90, seed=0):
    rng = np.random.default_rng(seed)
    gen, imp = np.asarray(genuine, dtype=np.float64), np.asarray(impostor, dtype=np.float64)
    far_list = []
    for _ in range(n_repeats):
        f, _ = compute_far_at_tar(rng.choice(gen, len(gen), replace=True),
                                   rng.choice(imp, len(imp), replace=True), target_tar, 10_000)
        far_list.append(f)
    arr = np.array(far_list)
    return float(arr.mean()), float(arr.std())

def format_far(v):
    if v == 0.0: return "0", "1/inf"
    return f"{v:.2e}", f"1/{int(round(1.0/v))}"

# Load data
all_users = load_all_users()
all_uids = sorted(all_users.keys())
n_total = len(all_uids)
print(f"Loaded {n_total} UV users")

# Adaptive split based on actual user count
if n_total >= 101:
    subset_base  = all_uids[:cfg.uv_baseline_n]
    val_add      = all_uids[cfg.uv_baseline_n:90]
    testfinal    = all_uids[90:101]
elif n_total >= 20:
    n_base  = int(n_total * 0.70)
    n_val   = int(n_total * 0.15)
    subset_base = all_uids[:n_base]
    val_add     = all_uids[n_base:n_base+n_val]
    testfinal   = all_uids[n_base+n_val:]
    print(f"Adaptive split: baseline={len(subset_base)}, val={len(val_add)}, test={len(testfinal)}")
else:
    raise ValueError(f"Too few UV users: {n_total}. Check UV preprocessing.")

assert len(testfinal) > 0, "testfinal is empty"

# Step 1: Baseline
print(f"\n=== Baseline (n={len(subset_base)}) ===")
seed_models, avs = [], []
for seed in range(cfg.n_training_runs):
    ckpt = os.path.join(CKPT_DIR, f"baseline_n{len(subset_base)}_seed{seed}.pt")
    if os.path.exists(ckpt):
        c = torch.load(ckpt, map_location=device)
        m = UVModel(n_classes=len(subset_base)).to(device)
        m.load_state_dict(c["model_state"]); seed_models.append(m); avs.append(c["acc"])
        print(f"  seed={seed}: loaded (acc={c['acc']:.1f}%)"); continue
    torch.manual_seed(seed); np.random.seed(seed)
    tr_f, vf, tf = {}, {}, {}
    for uid in subset_base:
        f = all_users[uid]; it, iv, ite = split_attempts(f, seed)
        tr_f[uid] = f[it]; vf[uid] = f[iv]; tf[uid] = f[ite]
    uid2cls = {u: i for i, u in enumerate(subset_base)}
    tr_lo = DataLoader(UVDataset(tr_f, uid2cls), cfg.batch_size, shuffle=True, drop_last=True)
    va_lo = DataLoader(UVDataset(vf, uid2cls), cfg.batch_size)
    model = UVModel(n_classes=len(subset_base)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.baseline_lr)
    loss_fn = TotalLoss(cfg.alpha_tm, cfg.supcon_temperature)
    best_acc, best_ckpt = 0, None
    for epoch in range(cfg.baseline_epochs):
        model.train()
        for Xb, yb in tr_lo:
            Xb, yb = Xb.to(device), yb.to(device); opt.zero_grad()
            logits, p1 = model(Xb, True); _, p2 = model(Xb, True)
            loss, _ = loss_fn(logits, torch.cat([p1, p2], 0), yb)
            loss.backward(); opt.step()
        model.eval(); correct = total = 0
        with torch.no_grad():
            for Xb, yb in va_lo:
                pred = model(Xb.to(device))[0].argmax(1).cpu()
                correct += (pred == yb).sum().item(); total += yb.size(0)
        acc = correct / total * 100
        if acc > best_acc:
            best_acc = acc; best_ckpt = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_ckpt)
    torch.save({"model_state": model.state_dict(), "acc": best_acc}, ckpt)
    seed_models.append(model); avs.append(best_acc)
    print(f"  seed={seed}: acc={best_acc:.1f}%")

chosen = int(np.argsort(avs)[len(avs)//2])
best_model = seed_models[chosen]
print(f"Chosen seed: {chosen}")

# Step 2: Fine-tune + test
print(f"\n=== Fine-tune + Bootstrap Test ({len(testfinal)} users) ===")
b_rows = [{"n_baseline": len(subset_base), "acc_val_mean": np.mean(avs), "acc_val_std": np.std(avs)}]
f_rows = []
for target_uid in testfinal:
    ft_ckpt = os.path.join(CKPT_DIR, f"ft_user{target_uid}_n{len(subset_base)}.pt")
    if os.path.exists(ft_ckpt):
        c = torch.load(ft_ckpt, map_location=device)
        ft = UVModel(n_classes=2).to(device); ft.load_state_dict(c["model_state"])
    else:
        torch.manual_seed(chosen); np.random.seed(chosen)
        ft = copy.deepcopy(best_model)
        for p in ft.branches.parameters(): p.requires_grad = False
        ft.head_a = nn.Linear(ft.embed_dim, 2).to(device); ft = ft.to(device)
        trainable = list(ft.head_a.parameters()) + list(ft.head_b.parameters()) + list(ft.siamese_proj.parameters())
        opt = torch.optim.Adam(trainable, lr=cfg.finetune_lr)
        crit = nn.CrossEntropyLoss()
        rng = np.random.default_rng(chosen); n_per = 150
        imp = np.concatenate([all_users[u][rng.choice(len(all_users[u]), max(1, n_per//len(subset_base)), replace=False)]
                              for u in subset_base], axis=0)[:n_per]
        own = all_users[target_uid][:n_per]
        X_mix = np.concatenate([imp, own]); y_mix = np.array([0]*len(imp) + [1]*len(own))
        shuf = rng.permutation(len(X_mix)); X_mix, y_mix = X_mix[shuf], y_mix[shuf]
        ft_lo = DataLoader(TensorDataset(
            torch.tensor(X_mix, dtype=torch.float32),
            torch.tensor(y_mix, dtype=torch.long)), 32, shuffle=True)
        val_imp = np.concatenate([all_users[u][:9] for u in val_add[:10]], 0) if val_add else imp[:90]
        val_gen = all_users[target_uid][:90]
        best_far, best_ft_ckpt = 1.0, None
        for epoch in range(cfg.finetune_epochs):
            ft.train()
            for Xb, yb in ft_lo:
                opt.zero_grad(); crit(ft(Xb.to(device))[0], yb.to(device)).backward(); opt.step()
            g, i = score_verification(ft, val_gen, val_imp)
            if len(g) > 0:
                fv = compute_far_at_tar(g, i, cfg.tar_threshold)[0]
                if fv < best_far:
                    best_far = fv; best_ft_ckpt = {k: v.clone() for k, v in ft.state_dict().items()}
        if best_ft_ckpt: ft.load_state_dict(best_ft_ckpt)
        torch.save({"model_state": ft.state_dict()}, ft_ckpt)
    gen = all_users[target_uid][:90]
    imp = np.concatenate([all_users[u][:9] for u in testfinal if u != target_uid], axis=0)[:90]
    g, i = score_verification(ft, gen, imp)
    if len(g) > 0:
        m, s = bootstrap_far(g, i, cfg.bootstrap_repeats, cfg.tar_threshold, seed=target_uid)
    else:
        m, s = 1.0, 0.0
    d, f = format_far(m)
    print(f"  user={target_uid}: FAR={m*100:.1f}+/-{s*100:.1f}% ({d}, {f})")
    f_rows.append({"user_id": target_uid, "n_baseline": len(subset_base), "far_mean": m, "far_std": s})
pd.DataFrame(b_rows).to_csv(os.path.join(RESULTS_DIR, "results_baseline.csv"), index=False)
pd.DataFrame(f_rows).to_csv(os.path.join(RESULTS_DIR, "results_uv_final.csv"), index=False)
print("\nUV training done.")

In [ ]:
# Cell 8: Results
print("=== MPI Results ===")
mpi_csv = os.path.join(RESULTS_DIR, "results_mpi.csv")
if os.path.exists(mpi_csv):
    df = pd.read_csv(mpi_csv)
    print(df.to_string(index=False))

print("\n=== UV Results ===")
uv_csv = os.path.join(RESULTS_DIR, "results_uv_final.csv")
if os.path.exists(uv_csv):
    df = pd.read_csv(uv_csv)
    print(df.to_string(index=False))

In [ ]:
# Cell 9: List outputs
for dirpath, _, filenames in os.walk("/kaggle/working"):
    for fname in filenames:
        fpath = os.path.join(dirpath, fname)
        rel = os.path.relpath(fpath, "/kaggle/working")
        print(f"  {rel:<60} {os.path.getsize(fpath)/1024:.1f} KB")
print("\nClick Save Version to persist.")